In [0]:
WITH
first_timers AS (
    SELECT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND cc_user_profile_id IS NOT NULL
    GROUP BY cc_user_profile_id
    HAVING COUNT(*) = 1
),
upgrades_took_effect AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_first_time AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    INNER JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_returning AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_total_signups AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_exits AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'annual1' AND cc_user_profile_id IS NOT NULL
      AND changed IS NOT TRUE
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
annual_direct_signup_starts AS (
    SELECT cc_user_profile_id, MIN(CAST(reporting_datetime AS DATE)) AS start_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND plan_id = 'annual1' AND reporting_datetime IS NOT NULL
    GROUP BY cc_user_profile_id
),
annual_upgrade_starts AS (
    SELECT cc_user_profile_id, MIN(CAST(reporting_datetime AS DATE)) AS start_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE AND reporting_datetime IS NOT NULL
    GROUP BY cc_user_profile_id
),
annual_starts AS (
    SELECT cc_user_profile_id, MIN(start_date) AS start_date
    FROM (SELECT * FROM annual_direct_signup_starts UNION ALL SELECT * FROM annual_upgrade_starts)
    GROUP BY cc_user_profile_id
),
annual_payments_window AS (
    SELECT DISTINCT cc_user_profile_id, reporting_datetime
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1'
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
annual_renewed AS (
    SELECT DISTINCT p.cc_user_profile_id
    FROM annual_payments_window AS p
    INNER JOIN annual_starts AS s ON p.cc_user_profile_id = s.cc_user_profile_id
    LEFT ANTI JOIN a_total_signups AS sg ON sg.cc_user_profile_id = p.cc_user_profile_id
    LEFT ANTI JOIN a_exits AS ex ON ex.cc_user_profile_id = p.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON ute.cc_user_profile_id = p.cc_user_profile_id
    WHERE DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date) >= 360
      AND (MOD(DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date), 365) <= 5
           OR MOD(DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date), 365) >= 360)
),
prev_window_snapshot AS (
    SELECT DISTINCT cc_user_profile_id, plan_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = '20251127'
),
g_purchasers AS (
    SELECT DISTINCT purchaser_id
    FROM cpx_dataanalytics_silver.cineclub.gifting
    WHERE CAST(from_utc_timestamp(CAST(purchase_date AS TIMESTAMP), 'America/New_York') AS DATE)
          BETWEEN '2025-11-28' AND '2025-12-01'
),
g_redeemers AS (
    SELECT DISTINCT redeemer_id
    FROM cpx_dataanalytics_silver.cineclub.gifting
    WHERE is_redeemed = TRUE
      AND CAST(from_utc_timestamp(CAST(redemption_date AS TIMESTAMP), 'America/New_York') AS DATE)
          BETWEEN '2025-11-28' AND '2025-12-01'
),
gift_purchasers_net_new AS (
    SELECT t.purchaser_id AS cc_user_profile_id FROM g_purchasers t
    LEFT ANTI JOIN prev_window_snapshot p ON t.purchaser_id = p.cc_user_profile_id
),
gift_redeemers_net_new AS (
    SELECT t.redeemer_id AS cc_user_profile_id FROM g_redeemers t
    LEFT ANTI JOIN prev_window_snapshot p ON t.redeemer_id = p.cc_user_profile_id
),
bf25_all_profile_ids AS (
    SELECT cc_user_profile_id, 'CPA' AS cost_bucket FROM a_first_time
    UNION ALL SELECT cc_user_profile_id, 'CPA' FROM a_returning
    UNION ALL SELECT cc_user_profile_id, 'CPA' FROM upgrades_took_effect
    UNION ALL SELECT cc_user_profile_id, 'CPR' FROM annual_renewed
    UNION ALL SELECT cc_user_profile_id, 'CPA' FROM gift_purchasers_net_new
    UNION ALL SELECT cc_user_profile_id, 'CPA' FROM gift_redeemers_net_new
),
cde_lookup_window AS (
    SELECT
        cc_user_profile_id,
        cde_id,
        ROW_NUMBER() OVER (
            PARTITION BY cc_user_profile_id
            ORDER BY ABS(DATEDIFF(TO_DATE(CAST(snapshot_date_id AS STRING), 'yyyyMMdd'), DATE'2025-12-01'))
        ) AS rn
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id BETWEEN '20251101' AND '20260101'
),
bf25_population AS (
    SELECT DISTINCT p.cc_user_profile_id, CAST(cl.cde_id AS STRING) AS cde_id, p.cost_bucket
    FROM bf25_all_profile_ids p
    JOIN cde_lookup_window cl ON p.cc_user_profile_id = cl.cc_user_profile_id
    WHERE cl.rn = 1
),
churn_check AS (
    SELECT
        bp.cost_bucket,
        bp.cde_id,
        CASE WHEN snap.renewal_status LIKE 'Active%' THEN 1 ELSE 0 END AS is_active
    FROM bf25_population bp
    LEFT JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot snap
        ON CAST(snap.cde_id AS STRING) = bp.cde_id
       AND snap.snapshot_date_id = '20260801'
)

SELECT
    cost_bucket,
    COUNT(DISTINCT cde_id) AS total_population,
    SUM(is_active) AS still_active,
    COUNT(DISTINCT cde_id) - SUM(is_active) AS churned,
    ROUND(SUM(is_active) * 100.0 / COUNT(DISTINCT cde_id), 2) AS retention_rate_pct
FROM churn_check
GROUP BY cost_bucket

UNION ALL

SELECT
    'TOTAL' AS cost_bucket,
    COUNT(DISTINCT cde_id),
    SUM(is_active),
    COUNT(DISTINCT cde_id) - SUM(is_active),
    ROUND(SUM(is_active) * 100.0 / COUNT(DISTINCT cde_id), 2)
FROM churn_check

ORDER BY CASE cost_bucket WHEN 'CPA' THEN 0 WHEN 'CPR' THEN 1 ELSE 2 END;

